# Klasik Görüntü İşleme ile Mikroskobik Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

**Veri seti:** Cell Tracking Challenge – Fluo-N2DH-GOWT1 (eğitim arşivi)  
**Seed:** 31092 (= 29066 + 2026)

Bu notebook projenin bütün adımlarını sırayla çalıştırır. Kodun tamamı `src/` klasöründeki modüllerdedir.

**Kilitli aşamalar** (veri seçimi, baseline, nihai pipeline, test tahminleri) yeniden *seçilmez*. Aynı kod tekrar çalıştırılır ve sonuçların kayıtlı dosyalarla **birebir aynı** olduğu doğrulanır.

> Veri seti repoda yoktur. Zip dosyasını indirip `data/Fluo-N2DH-GOWT1/` altına açın (README'ye bakın).  
> Hücre görüntüsü içeren şekiller `report_figures/` klasörüne yazılır ve repoya yüklenmez. Notebook çıktıları da GitHub'a yüklenmeden önce temizlenmelidir.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
from common import DATA_ROOT, RESULTS, ROOT
from reproduce import run_main, check_selection, check_validation, check_locked_test, check_experiments

assert (DATA_ROOT / "01").is_dir() and (DATA_ROOT / "02").is_dir(), \
    f"Veri seti bulunamadı: {DATA_ROOT}"
pd.set_option("display.width", 200)

# Bütün deneyleri (118 konfigürasyon) yeniden çalıştırmak birkaç dakika sürer.
RUN_EXPERIMENTS = True

## 1. Veri seti incelemesi (Bölüm 2)

In [ ]:
import explore_data
explore_data.main()
pd.read_csv(RESULTS / "data_sequence_summary.csv")

## 2. Geliştirme, doğrulama ve test görüntülerinin seçimi (Bölüm 3)

In [ ]:
sel = check_selection()
sel.groupby("group").frame_index.apply(list)

## 3. Referans hücre sayıları – yalnızca geliştirme + doğrulama (Bölüm 4)

In [ ]:
import reference_counts
run_main(reference_counts)
pd.read_csv(RESULTS / "reference_counts_devval.csv")

## 4. Başlangıç yöntemi – Otsu + bağlı bileşen analizi (Bölüm 5)

Minimum alan (50 px) geliştirme setinde seçildi ve `results/baseline_config.json` ile sabitlendi.

In [ ]:
import baseline
run_main(baseline)  # --tune verilmez: sabitlenmiş ayar kullanılır
pd.read_csv(RESULTS / "baseline_devval_summary.csv")

## 5. Kontrollü deneyler – yoğunluk, filtre, segmentasyon, morfoloji, watershed (Bölüm 6–10)

In [ ]:
if RUN_EXPERIMENTS:
    exp = check_experiments()
else:
    exp = pd.read_csv(ROOT / "experiments.csv")
exp[["experiment_id", "stage", "changed", "MAE", "RMSE", "Bias",
     "missed", "extra_from_split", "merged", "false_pos"]].head(20)

In [ ]:
print(json.dumps(json.loads((RESULTS / "candidate_pipelines.json").read_text()), indent=1, ensure_ascii=False))

In [ ]:
import make_experiment_figures
make_experiment_figures.main()

## 6. Doğrulama ve nihai yöntem seçimi (Bölüm 11)

In [ ]:
check_validation()

In [ ]:
import make_pipeline_figure
make_pipeline_figure.main()
json.loads((ROOT / "final_pipeline.json").read_text())["pipeline"]

## 7. Kilitli test tahminleri (Bölüm 12)

In [ ]:
check_locked_test()

## 8. Test sonuçları (Bölüm 13)

In [ ]:
import evaluate_test
evaluate_test.main()
pd.read_csv(RESULTS / "test_summary.csv")

## 9. Zamansal analiz (Bölüm 14)

In [ ]:
import temporal_analysis
temporal_analysis.main()